# Build circuits in Fortran, simulate in Python

## Prerequisites

* Python 3.10+ with this branch's built `qdk`, `ipykernel`, and IPython in the selected Python kernel, plus JupyterLab or VS Code's Python/Jupyter extensions.
* A checkout of this branch, Rust/Cargo, a GCC/Clang-compatible C compiler, and a Fortran compiler with `ISO_C_BINDING` support (LLVM Flang or GNU Fortran). Set `CC` and `FC` to select compilers if needed.
* The matching native SDK, including the Fortran binding sources, library, and header. From the repository root, run `python source/circuit_capi/build.py` after completing the [setup guide](README.md). The module is compiled with your chosen compiler; no prebuilt `.mod` file is required.
* macOS or Linux; use WSL2 for this native sample on Windows.
* Optional GPU execution requires a QDK-supported GPU and drivers. It is disabled by default.

This is a Python-kernel notebook. It compiles and runs a standalone Fortran program, then Python loads and simulates its exported circuits. A Fortran Jupyter kernel and Python embedding in the Fortran program are not required. Simulation uses compiled QIR because the circuit API has no direct run method.

In [ ]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
from IPython.display import Markdown, display
from qdk import Result, TargetProfile
from qdk.circuit import QuantumCircuit
from qdk.simulation import run_qir

root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "source/circuit_capi/includes/qdk_circuit.h").is_file()
    ),
    None,
)
if root is None:
    raise RuntimeError("Start this kernel inside the QDK checkout.")
sys.path.insert(0, str(root / "samples/notebooks/circuit_api"))
from _native import SOURCES, build_and_run

RUN_GPU = False
SHOTS = 128

## Construct a Bell circuit and a feedback circuit

The [Fortran program](native/circuits.f90) uses the reusable `qdk_circuit` module and type-bound procedures. It clones a shared H preparation into two independent circuits, then closes the original. Qubit and clbit indices remain **zero-based**, even in Fortran.

* The Base circuit uses H, CX, and two terminal measurements.
* The Adaptive circuit measures qubit 0 and opens an if-body applying X to qubit 1 when that bit is One.
* A deliberate Base export of the feedback circuit checks the diagnostic, then an Adaptive export succeeds. This demonstrates capability checking, not an ignored error.

The [reusable binding](../../../source/circuit_capi/bindings/fortran/README.md) hides `c_ptr`, descriptors, diagnostics, and native buffer ownership. Calls return an `error_t`; the application decides how to handle failures. `if_one` scopes a `body_t` callback, finishes or cancels its builder, and releases its condition. This sample uses a module-procedure callback for compatibility with macOS Flang.

Circuits require explicit `close` calls. Assignment aliases the same circuit; use `clone` for an independent owner. Closing invalidates all aliases, and a retained body is rejected after its callback ends. The binding itself never calls `error stop`; that policy belongs to the sample application.

Edit [circuits.f90](native/circuits.f90), then rerun the compile cell. Binary snapshots are written using unformatted stream I/O, preserving embedded NUL bytes without Fortran record markers.

In [ ]:
source = (SOURCES / "circuits.f90").read_text(encoding="utf-8")
display(Markdown(f"```fortran\n{source}\n```"))

In [ ]:
work = TemporaryDirectory(prefix="qdk-fortran-circuits-")
directory = Path(work.name).resolve()
print("Artifact directory:", directory)
build_and_run("fortran", directory)
print("Native artifacts (removed by the final cleanup cell):")
for path in sorted(directory.iterdir()):
    print(path)

## Load snapshots, verify native QIR, and run on the CPU

The Fortran process has freed all its handles before Python reads these files. The typed `.qdk` snapshots are self-contained circuit documents, not JSON or HUGR documents. Each has a readable `.sexpr` snapshot alongside it; below we verify that both formats restore the same circuit. Python's compiled QIR must match the native `.ll` export for the same target profile.

Both examples yield only `00` and `11`. The Bell circuit creates quantum correlations; the adaptive circuit uses classical feedback. We check the outcome invariant and shot count, not an exact 50/50 split.

In [ ]:
programs = {}
for name, profile in [
    ("base", TargetProfile.Base),
    ("adaptive", TargetProfile.Adaptive),
]:
    payload = (directory / f"{name}.qdk").read_bytes()
    circuit = QuantumCircuit.from_bytes(payload)
    assert circuit.to_bytes() == payload
    text = (directory / f"{name}.sexpr").read_text(encoding="utf-8")
    assert QuantumCircuit.from_text(text).to_bytes() == payload
    programs[name] = circuit.to_qir(profile)
    assert programs[name] == (directory / f"{name}.ll").read_text(encoding="utf-8")
    print(name, "\n", circuit)


def correlated_counts(samples):
    assert len(samples) == SHOTS
    assert all(
        len(s) == 2 and s[0] == s[1] and s[0] in (Result.Zero, Result.One)
        for s in samples
    )
    return Counter(
        "".join("1" if bit == Result.One else "0" for bit in s) for s in samples
    )


for name, qir in programs.items():
    samples = run_qir(qir, shots=SHOTS, seed=42, type="cpu")
    print(name, "CPU:", correlated_counts(samples))

## Opt in to GPU execution

Set `RUN_GPU = True` in the setup cell and rerun this cell after the CPU section. `type='gpu'` requests the GPU explicitly and does not fall back to CPU. GPU initialization failures remain visible. CPU/GPU shot sequences may differ with the same seed; these small programs demonstrate interoperability rather than GPU speedup.

In [ ]:
if RUN_GPU:
    for name, qir in programs.items():
        samples = run_qir(qir, shots=SHOTS, seed=42, type="gpu")
        print(name, "GPU:", correlated_counts(samples))
else:
    print("GPU execution disabled; set RUN_GPU = True to enable it.")

## Run the native QIR directly

Snapshot loading is useful when Python needs to inspect or transform a circuit. If it only needs to run the already compiled program, `run_qir` can consume native QIR directly. Read textual `.ll` files as strings; byte inputs to `run_qir` are reserved for LLVM bitcode. This decouples a Fortran circuit producer from its simulator process.

In [ ]:
native_qir = (directory / "adaptive.ll").read_text(encoding="utf-8")
samples = run_qir(native_qir, shots=SHOTS, seed=42, type="cpu")
print("Native Adaptive QIR on CPU:", correlated_counts(samples))

## Cleanup

Copy snapshots or QIR outside `directory` first if you want to retain them. This cell removes only the notebook's temporary compiler outputs and generated circuit files. The Fortran source and binding module remain in the checkout.

In [ ]:
work.cleanup()
assert not directory.exists()